In [1]:
import pandas as pd
import numpy as np

In [2]:
NAMES = ['movie', 'Amazon-KG-5core-Movies_and_TV']

In [3]:
def build_graph_and_interactions(interactions: pd.DataFrame, link: pd.DataFrame, static_graph: pd.DataFrame):
    #########################################################
    #### 1. Build static knowledge graph
    #########################################################
    user_tokens = interactions['user_id:token'].unique()
    num_users = len(user_tokens)

    entity_tokens = link['entity_id:token'].unique()

    head_tokens = static_graph['head_id:token'].unique()
    tail_tokens = static_graph['tail_id:token'].unique()

    all_entity_tokens = pd.unique(np.concatenate([entity_tokens, head_tokens, tail_tokens]))
    entity2id = {entity: (num_users + idx + 1) for idx, entity in enumerate(all_entity_tokens)}

    static_graph['head_id'] = static_graph['head_id:token'].map(entity2id)
    static_graph['tail_id'] = static_graph['tail_id:token'].map(entity2id)

    static_graph['relation_id'] = static_graph['relation_id:token'].astype('category').cat.codes  
    static_graph['relation_id'] = static_graph['relation_id'] + 1   

    static_graph = static_graph.dropna()
    static_graph = static_graph.astype({'head_id': 'long', 'relation_id': 'long', 'tail_id': 'long'})
    static_graph = static_graph[['head_id', 'relation_id', 'tail_id',
             'head_id:token', 'relation_id:token', 'tail_id:token']]

    static_graph = static_graph.sort_values(by=['head_id', 'tail_id'])

    #########################################################
    #### 2. Build interactions
    #########################################################
    user2id = {user: (idx + 1) for idx, user in enumerate(user_tokens)}

    item2entity = dict(zip(link['item_id:token'], link['entity_id:token']))
    item2entity_id = {item: entity2id[entity] for item, entity in item2entity.items()}

    interactions['entity_id:token'] = interactions['item_id:token'].map(item2entity)
    interactions['user_id'] = interactions['user_id:token'].map(user2id)
    interactions['entity_id'] = interactions['item_id:token'].map(item2entity_id)

    interactions = interactions.dropna()
    interactions = interactions.astype({'user_id': 'long', 'entity_id': 'long'})
    interactions = interactions[['user_id', 'entity_id', 'timestamp','user_id:token',
                                 'entity_id:token', 'item_id:token']]

    interactions = interactions.sort_values(by=['entity_id'])

    return static_graph, interactions

In [ ]:
def split_data(df, user_col='user_id', time_col='timestamp'):
    df = df.sort_values(by=[user_col, time_col]).reset_index(drop=True)

    user_counts = df.groupby(user_col).size()

    valid_users = user_counts[user_counts >= 5].index
    df = df[df[user_col].isin(valid_users)]

    train_list, val_list, test_list = [], [], []

    for user, user_df in df.groupby(user_col):
        n = len(user_df)
        
        train_idx = int(n * 0.7)
        val_idx = int(n * 0.8)

        train_list.append(user_df.iloc[:train_idx])
        val_list.append(user_df.iloc[train_idx:val_idx])
        test_list.append(user_df.iloc[val_idx:])

    # 4. Gộp lại thành 3 tập dữ liệu hoàn chỉnh
    train_df = pd.concat(train_list).reset_index(drop=True)
    val_df = pd.concat(val_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, val_df, test_df


In [5]:
if __name__ == '__main__':
    name0 = NAMES[0]
    name1 = NAMES[1]

    interactions = pd.read_csv(f'./data/{name0}/{name0}_interaction.csv', sep= ',')
    link = pd.read_csv(f'./data/{name0}/{name1}.link', sep="\t")
    static_graph = pd.read_csv(f'./data/{name0}/{name1}.kg', sep="\t")


    static_graph, interactions = build_graph_and_interactions(interactions, link, static_graph)

    static_graph.to_csv(f'./data/{name0}/{name0}_processed_static_graph.csv', index=False)
    interactions.to_csv(f'./data/{name0}/{name0}_processed_interactions.csv', index= False)

    train_df, val_df, test_df = split_data(interactions)
    train_df = train_df.sort_values(by=['user_id', 'entity_id'])
    val_df = val_df.sort_values(by=['user_id', 'entity_id'])
    test_df = test_df.sort_values(by=['user_id', 'entity_id'])

    train_df.to_csv(f'./data/{name0}/{name0}_train_interactions.csv', index= False)
    val_df.to_csv(f'./data/{name0}/{name0}_val_interactions.csv', index= False)
    test_df.to_csv(f'./data/{name0}/{name0}_test_interactions.csv', index= False)
